# RAG system based on football articles

In [1]:
from pathlib import Path
import time
from openai import OpenAI
from dotenv import load_dotenv
import os
from typing import Any, Dict, List, Optional

load_dotenv()
OPEN_AI_API_KEY = os.getenv("OPEN_AI_API_KEY")
OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL")

# instantiate the openAI client
client = OpenAI(api_key=OPEN_AI_API_KEY)

## Creation of the knowledge base
We first need to upload our file, specifying its purpose. The file can then be used across various endpoints.

Once the file is uploaded, it should be added to a [Vector Store](https://platform.openai.com/docs/api-reference/vector-stores), where it gets broken into smaller chunks. The chunked-up file consitutes the knowledge base of our RAG system.

In [2]:
def _list_all_vector_stores(client, limit=100):
    """Paginate through all vector stores (up to a few hundred)."""
    stores = []
    cursor = None
    while True:
        resp = client.vector_stores.list(limit=limit, after=cursor) if cursor else client.vector_stores.list(limit=limit)
        stores.extend(getattr(resp, "data", []) or [])
        cursor = getattr(resp, "last_id", None)
        if not cursor or len(getattr(resp, "data", []) or []) < limit:
            break
    return stores

def _find_vector_store_by_name(client, store_name: str):
    """Return the most recent store with the given name, or None if it does not exist."""
    stores = _list_all_vector_stores(client)
    same_name = [vs for vs in stores if getattr(vs, "name", None) == store_name]
    if not same_name:
        return None
    # If multiple stores share the same name, pick the most recent by created_at (if available)
    same_name.sort(key=lambda x: getattr(x, "created_at", 0), reverse=True)
    return same_name[0]

def _list_store_files_with_names(client, vector_store_id: str, limit=100):
    """Return a list of tuples [(file_id, filename, status)] for files attached to the store."""
    items = []
    cursor = None
    while True:
        resp = client.vector_stores.files.list(
            vector_store_id=vector_store_id, limit=limit, after=cursor
        ) if cursor else client.vector_stores.files.list(
            vector_store_id=vector_store_id, limit=limit
        )
        data = getattr(resp, "data", []) or []
        for it in data:
            fid = getattr(it, "id", None) or getattr(it, "file_id", None)
            status = getattr(it, "status", None)
            # Retrieve filename from Files metadata (fallbacks for SDK differences)
            try:
                fmeta = client.files.retrieve(fid)
                fname = getattr(fmeta, "filename", None) or getattr(fmeta, "name", None)
            except Exception:
                fname = None
            items.append((fid, fname, status))
        cursor = getattr(resp, "last_id", None)
        if not cursor or len(data) < limit:
            break
    return items

def _wait_file_indexed(client, vector_store_id: str, file_id: str, timeout_s=180, poll_every_s=2):
    """Block until the file is indexed (status='completed') or raise on failure/timeout."""
    start = time.time()
    while True:
        cur = client.vector_stores.files.retrieve(vector_store_id=vector_store_id, file_id=file_id)
        status = getattr(cur, "status", None)
        if status in ("completed", "failed", "cancelled"):
            if status != "completed":
                raise RuntimeError(f"Indexing did not complete (status={status}) for file_id={file_id}.")
            return
        if time.time() - start > timeout_s:
            raise TimeoutError(f"Timeout while waiting for indexing of file_id={file_id}.")
        time.sleep(poll_every_s)

def upload_file_to_vector_store(path, store_name="Some Fake Football Articles", wait_index=True):
    """
    Upload 'path' to the vector store 'store_name' in an idempotent way:
    - Reuse the store if it already exists (by name).
    - Do not re-attach the file if it is already there (by filename).
    - Optionally wait for indexing to complete (wait_index=True).
    Returns (vector_store, file_id_used_or_existing).
    """
    p = Path(path)
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f"File not found: {p.resolve()}")

    # 1) Get or create the vector store
    vector_store = _find_vector_store_by_name(client, store_name)
    if vector_store is None:
        vector_store = client.vector_stores.create(name=store_name)

    # 2) Check if a file with the same name is already attached
    current_files = _list_store_files_with_names(client, vector_store.id)
    existing = [fid for (fid, fname, _status) in current_files if fname == p.name]

    if existing:
        file_id = existing[0]
        # Ensure the file is fully indexed before returning (optional)
        if wait_index:
            _wait_file_indexed(client, vector_store.id, file_id)
        return vector_store, file_id

    # 3) Upload and attach only if it does not exist yet
    with open(p, "rb") as fh:
        file_obj = client.files.create(file=fh, purpose="assistants")

    link = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file_obj.id
    )

    # 4) Optionally wait until the file is fully indexed
    if wait_index:
        _wait_file_indexed(client, vector_store.id, getattr(link, "id", None) or file_obj.id)

    return vector_store, file_obj.id


vector_store, file_id = upload_file_to_vector_store(
    "data/fake_football_articles.json",
    store_name="Some Fake Football Articles",
    wait_index=True
)
print("Vector store:", vector_store.id)
print("File id:", file_id)


Vector store: vs_68edc24f143c8191a6abeae68581b6e8
File id: file-WXSZ76xgjujCZXe6HZB2ZC


## Build a very simple Q&A flow

Once our knowledge base is created, we can start interacting with our RAG, using the [Responses API](https://platform.openai.com/docs/api-reference/responses).

The function `generate_response()` receives a prompt from the user as a parameter, and returns an answer using the [model](https://platform.openai.com/docs/models) specified in the `.env` file based on the knowledge base just created.

It does so by using `OpenAI.responses.create()` ([official docs](https://platform.openai.com/docs/api-reference/responses/create)). This function can be extended to:
- set how many chunks are used for the response with the `max_num_results` in the `file_search` tool. Setting it to a lower number may prevent hallucinations.
- add support for web search by specifying a new tool type called `web_search_preview`.
- specify additional output data to include in the model response with the `include` parameter. For example, `include=["output[*].file_search_call.search_results"]` will show what chunks were used to generate the answer and what confidence score was given to a particular chunk.

In [3]:
def _store_has_indexed_files(client, vector_store_id: str) -> bool:
    """
    Returns True if the vector store has at least one file with status 'completed'.
    """
    try:
        page = client.vector_stores.files.list(vector_store_id=vector_store_id, limit=50)
        for it in getattr(page, "data", []) or []:
            status = getattr(it, "status", None)
            if status == "completed":
                return True
        return False
    except Exception:
        return False

def _extract_text_from_response(resp_obj) -> str:
    """
    Extracts text from the SDK response defensively, concatenating all text blocks
    in 'message' outputs (some SDK versions structure content differently).
    """
    chunks: List[str] = []
    output = getattr(resp_obj, "output", []) or []

    for item in output:
        if getattr(item, "type", None) != "message":
            continue
        content_list = getattr(item, "content", []) or []
        for block in content_list:
            
            txt = getattr(block, "text", None)
            if txt:
                chunks.append(str(txt))
    return " ".join(chunks).strip()

def _retry_call(fn, max_retries=3, base_sleep=1.0, exceptions=(Exception,)):
    """
    Simple retry helper with exponential backoff.
    """
    attempt = 0
    while True:
        try:
            return fn()
        except exceptions as e:
            attempt += 1
            if attempt > max_retries:
                raise
            time.sleep(base_sleep * (2 ** (attempt - 1)))

def generate_response(
    prompt: str,
    *,
    client,
    model: str,
    vector_store_id: str,
    temperature: float = 0.2,
    max_output_tokens: Optional[int] = None,
    require_indexed: bool = True,
    structured_sections: bool = True,
    return_meta: bool = True,
) -> Dict[str, Any]:
    """
    RAG call using OpenAI Responses + File Search (Vector Stores).

    Args:
        prompt: User question (e.g., "Is Thierry Doumbia a good player?")
        client: OpenAI client (already initialized)
        model: Model name (e.g., OPEN_AI_MODEL)
        vector_store_id: Existing vector store id (idempotent workflow)
        temperature: Lower = more deterministic
        max_output_tokens: Optional cap for completion length
        require_indexed: If True, validates at least one file is fully indexed
        structured_sections: If True, ask the model to answer with fixed sections
        return_meta: If True, returns dict with text + meta; else only text in 'text'

    Returns:
        dict with:
            - status: "ok" | "no_articles" | "error" | "not_ready"
            - text: final answer text
            - sections: parsed structure (best-effort) if structured_sections=True
            - meta: model, vector_store_id, timing (best-effort), etc.
    """

    # 0) Pre-checks (fast fail)
    if not vector_store_id:
        return {"status": "error", "text": "", "error": "Missing vector_store_id"}

    if require_indexed and not _store_has_indexed_files(client, vector_store_id):
        return {"status": "not_ready", "text": "", "error": "Vector store has no completed indexed files yet"}

    # 1) Build instructions
    base_instr = (
        "You need to perform a file search against the provided vector store and summarise "
        "all the articles you can find about the player provided by the user. "
        "If there are none, respond with exactly: No articles on the provided player. "
        "Focus on strengths and weaknesses, and on how the judgement evolved over time if multiple articles exist. "
        "Use only information grounded in the retrieved files; do not invent sources."
    )

    if structured_sections:
        base_instr += (
            " Answer in the following structure:\n"
            "Strengths: <bulleted or short sentences>\n"
            "Weaknesses: <bulleted or short sentences>\n"
            "EvolutionOverTime: <chronological summary, dates if available>\n"
            "Sources: <titles or identifiers of the articles used>\n"
            "If no articles exist, reply exactly: No articles on the provided player"
        )

    # 2) Prepare kwargs for the API call (supporting both tools + tool_resources styles)
    call_kwargs = {
        "model": model,
        "input": prompt,
        "instructions": base_instr,
        "temperature": temperature,
        # Legacy style: vector_store_ids dentro de 'tools'
        "tools": [
            {
                "type": "file_search",
                "vector_store_ids": [vector_store_id]
            }
        ],
    }
    if max_output_tokens is not None:
        call_kwargs["max_output_tokens"] = max_output_tokens


    # 3) Call with simple retries (handles transient errors/rate limits)
    t0 = time.time()
    resp = _retry_call(lambda: client.responses.create(**call_kwargs), max_retries=3, base_sleep=1.0)
    elapsed_s = round(time.time() - t0, 3)

    # 4) Extract text safely
    text = _extract_text_from_response(resp)

    # 5) Determine status
    norm = (text or "").strip()
    if norm == "No articles on the provided player":
        status = "no_articles"
    elif not norm:
        status = "error"
    else:
        status = "ok"

    # 6) (Best-effort) parse sections if requested
    sections = None
    if structured_sections and norm:
        def _grab(block_name: str, txt: str) -> Optional[str]:
            # naive split by header name
            lowered = txt
            anchor = block_name + ":"
            if anchor in lowered:
                part = lowered.split(anchor, 1)[1].strip()
                for nxt in ["Strengths:", "Weaknesses:", "EvolutionOverTime:", "Sources:"]:
                    if nxt != anchor and nxt in part:
                        part = part.split(nxt, 1)[0].strip()
                return part or None
            return None

        sections = {
            "Strengths": _grab("Strengths", norm),
            "Weaknesses": _grab("Weaknesses", norm),
            "EvolutionOverTime": _grab("EvolutionOverTime", norm),
            "Sources": _grab("Sources", norm),
        }

    # 7) Build return payload
    out: Dict[str, Any] = {
        "status": status,
        "text": norm,
    }
    if structured_sections:
        out["sections"] = sections
    if return_meta:
        # usage fields vary across SDK versions; guard access
        usage = getattr(resp, "usage", None)
        out["meta"] = {
            "model": model,
            "vector_store_id": vector_store_id,
            "elapsed_s": elapsed_s,
            "usage": {
                "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
                "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
                "total_tokens": getattr(usage, "total_tokens", None) if usage else None,
            }
        }
    return out

In [4]:
result = generate_response(
    prompt="Is Thierry Doumbia a good player?",
    client=client,
    model=OPEN_AI_MODEL,
    vector_store_id=vector_store.id,
    temperature=0.2,
    max_output_tokens=800,
    require_indexed=True,
    structured_sections=True,
    return_meta=True
)

print(result["status"])
print(result["text"])
if result.get("sections"):
    print("Strengths:", result["sections"].get("Strengths"))
    print("Weaknesses:", result["sections"].get("Weaknesses"))
    print("EvolutionOverTime:", result["sections"].get("EvolutionOverTime"))
    print("Sources:", result["sections"].get("Sources"))


ok
### Strengths:
- Exceptional acceleration, making him difficult for defenders to handle.
- Raw pace that could be an asset in faster leagues.

### Weaknesses:
- Lacks technical finesse, leading to turnovers in tight spaces.
- His athleticism does not compensate for his technical limitations.

### Evolution Over Time:
- **January 6, 2025**: Initial praise for his acceleration and impact on the field, described as a nightmare for defenders.
- **January 30, 2025**: Criticism emerges regarding his technical skills, highlighting his inability to maintain possession under pressure.
- **February 16, 2025**: Reports of interest from clubs in the Zandora League, indicating that despite his limitations, his speed remains a valuable asset.

### Sources:
- "Doumbia’s Acceleration"
- "Doumbia’s Limitations"
- "Doumbia Links"
Strengths: - Exceptional acceleration, making him difficult for defenders to handle.
- Raw pace that could be an asset in faster leagues.

###
Weaknesses: - Lacks technical 